In [0]:
%pip install --upgrade "pyiceberg[s3fs]" typing_extensions "pyarrow>=16.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 185.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.4/781.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 164.2 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-eac1f520-971b-4b10-8258-ed67ef7d20de
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 21.0.0
    Not uninstalling pyarrow at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-eac1f520-971b-4b10-8258-ed67ef7d20de
    Can't uninstall 'pyarrow'. No files were found to

In [0]:
dbutils.library.restartPython()

In [0]:
from pyiceberg.catalog import load_catalog
from pyiceberg import __version__
import pyarrow.types
print("Version of pyiceberg: " + __version__)

# Compatibility shim: PyIceberg 0.11.1 requires pa.types.is_string_view (PyArrow 16+)
if not hasattr(pyarrow.types, 'is_string_view'):
    pyarrow.types.is_string_view = lambda t: False

# 1. Fetch credentials
polaris_oauth_client_id = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_client_id")
polaris_oauth_client_secret = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_client_secret")
polaris_oauth_token_url = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_token_url")
polaris_oauth_scope = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_scope")
polaris_base_url = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_base_url")
polaris_warehouse = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_warehouse")

aws_access_key = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_access_key")
aws_secret_key = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_secret_key")
aws_region = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_region")

# 2. Connect to Polaris Iceberg catalog
catalog = load_catalog(
    "polaris",
    **{
        "type": "rest",
        "uri": polaris_base_url,
        "oauth.token.url": polaris_oauth_token_url,
        "credential": f"{polaris_oauth_client_id}:{polaris_oauth_client_secret}",
        "scope": polaris_oauth_scope,
        "warehouse": polaris_warehouse,
        "s3.access-key-id": aws_access_key,
        "s3.secret-access-key": aws_secret_key,
        "s3.region": aws_region,
        "py-io-impl": "pyiceberg.io.fsspec.FsspecFileIO"
    }
)

catalog.list_namespaces()
catalog._session.headers.pop("X-Iceberg-Access-Delegation", None)

print("Polaris catalog connected successfully.")

Version of pyiceberg: 0.12.0
Polaris catalog connected successfully.


In [0]:
# 3. Discover all namespaces and tables in the catalog
all_tables = []
for ns in catalog.list_namespaces():
    ns_name = ns[0] if isinstance(ns, tuple) else ns
    all_tables.extend(catalog.list_tables(ns_name))

print(f"Found {len(all_tables)} table(s): {[f'{t[0]}.{t[1]}' for t in all_tables]}")

Found 40 table(s): ['cdc_alarm.alarm', 'cdc_event.c8y_Position', 'cdc_event.event', 'cdc_event.jobStatus', 'cdc_inventory.c8y_ActiveAlarmsStatus', 'cdc_inventory.c8y_DeviceSimulator', 'cdc_inventory.c8y_Position', 'cdc_inventory.c8y_ua_Node', 'cdc_inventory.com_cumulocity_opcua_common_model_mapping_DeviceType', 'cdc_inventory.inventory', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D10', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D11', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D12', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D13', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D15', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D16', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D20', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D7', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D8', 'cdc_inventory.opcuaVal_ns_x003D2_x003Bi_x003D9', 'cdc_measurement.c8y_Temperature', 'cdc_operation.operation', 'cdc_rejected.trash', 'internal.checkpoint_store', 'latest_inventory.c8y_ActiveAlarmsStatus

In [0]:
# 4. Incremental import using PyIceberg exclusively for reading Iceberg data
import json
import time
import numpy as np
import pandas as pd_lib
from decimal import Decimal
from datetime import datetime, timezone, date
from pyspark.sql.functions import monotonically_increasing_id, lit
from pyspark.sql.types import StringType, LongType
from pyiceberg.types import TimestamptzType
from pyiceberg.expressions import And, GreaterThan, LessThanOrEqual

# Ensure snapshot tracking table exists
spark.sql("""
    CREATE TABLE IF NOT EXISTS _iceberg_snapshot_tracking (
        source_table STRING, snapshot_id STRING, sequence_number BIGINT, updated_at TIMESTAMP
    ) USING DELTA
""")
print("Snapshot tracking table ready.")

run_end_time = datetime.now(timezone.utc)
print(f"Run end time (upper bound): {run_end_time.isoformat()}")


def safe_json_dumps(x):
    """Safely serialize to JSON, handling special types."""
    if x is None:
        return None
    try:
        if pd_lib.isna(x):
            return None
    except (ValueError, TypeError):
        pass
    if isinstance(x, np.ndarray):
        x = x.tolist()
    return json.dumps(x, default=lambda o:
        float(o) if isinstance(o, (Decimal, np.floating)) else
        int(o) if isinstance(o, np.integer) else
        o.isoformat() if isinstance(o, (datetime, date)) else
        o.tolist() if isinstance(o, np.ndarray) else str(o)
    )


def flatten_pandas_df(pdf):
    """Expand dict columns into flat columns; serialize lists/complex types to JSON."""
    cols_to_drop = []
    new_cols = {}

    for col_name in list(pdf.columns):
        if pdf[col_name].dtype != object:
            continue
        sample = pdf[col_name].dropna().head(10)
        if len(sample) == 0:
            continue
        first_val = sample.iloc[0]

        if isinstance(first_val, dict):
            keys_set = set()
            for val in sample:
                if isinstance(val, dict):
                    keys_set.update(val.keys())
            if len(keys_set) <= 20:
                cols_to_drop.append(col_name)
                for key in sorted(keys_set):
                    series = pdf[col_name].apply(lambda x, k=key: x.get(k) if isinstance(x, dict) else None)
                    if series.apply(lambda x: isinstance(x, Decimal)).any():
                        series = series.apply(lambda x: float(x) if isinstance(x, Decimal) else x)
                    new_cols[f"{col_name}_{key}"] = series
            else:
                pdf[col_name] = pdf[col_name].apply(safe_json_dumps)
        elif isinstance(first_val, (list, np.ndarray)):
            pdf[col_name] = pdf[col_name].apply(safe_json_dumps)
        elif isinstance(first_val, (datetime, date)):
            pdf[col_name] = pdf[col_name].apply(lambda x: x.isoformat() if isinstance(x, (datetime, date)) else x)

    if cols_to_drop:
        pdf = pdf.drop(columns=cols_to_drop)
    for name, series in new_cols.items():
        sample = series.dropna().head(5)
        if len(sample) > 0:
            fv = sample.iloc[0]
            if isinstance(fv, (list, np.ndarray)):
                series = series.apply(safe_json_dumps)
            elif isinstance(fv, (datetime, date)):
                series = series.apply(lambda x: x.isoformat() if isinstance(x, (datetime, date)) else x)
        pdf[name] = series
    return pdf


def get_watermark(target, time_col):
    """Get max timestamp from existing Delta table as ISO string."""
    try:
        max_ts = spark.sql(f"SELECT MAX(`{time_col}`) as max_ts FROM {target}").collect()[0]["max_ts"]
        return max_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] if max_ts else None
    except Exception:
        return None


def get_stored_snapshot(source):
    """Retrieve last-imported snapshot ID and sequence number."""
    try:
        row = spark.sql(f"SELECT snapshot_id, sequence_number FROM _iceberg_snapshot_tracking WHERE source_table = '{source}'").collect()
        return (row[0]["snapshot_id"], row[0]["sequence_number"]) if row else (None, None)
    except Exception:
        return (None, None)


def _read_new_files(iceberg_table, stored_seq):
    """Fallback: read new Parquet data files directly via PyArrow (bypasses equality delete limitation)."""
    import pyarrow as pa
    import pyarrow.parquet as pq

    entries_df = iceberg_table.inspect.entries().to_pandas()
    # content and file_path are nested inside the 'data_file' dict column
    entries_df['_content'] = entries_df['data_file'].apply(lambda x: x.get('content', -1))
    entries_df['_file_path'] = entries_df['data_file'].apply(lambda x: x.get('file_path', ''))
    # Filter for new DATA files (status=1=ADDED, content=0=DATA)
    if stored_seq is not None:
        mask = (entries_df['sequence_number'] > stored_seq) & (entries_df['status'] == 1) & (entries_df['_content'] == 0)
    else:
        mask = entries_df['_content'] == 0
    new_files = entries_df[mask]['_file_path'].tolist()

    if not new_files:
        return pd_lib.DataFrame()

    print(f"  Reading {len(new_files)} new file(s) directly via PyArrow")
    io = iceberg_table.io
    tables = []
    for fp in new_files:
        input_file = io.new_input(fp)
        with input_file.open() as f:
            tables.append(pq.read_table(f))
    return pa.concat_tables(tables).to_pandas()


def store_snapshot(source, snapshot_id, seq_num=None):
    """Persist snapshot ID and sequence number."""
    seq_val = seq_num if seq_num is not None else "NULL"
    spark.sql(f"""
        MERGE INTO _iceberg_snapshot_tracking AS t
        USING (SELECT '{source}' AS source_table, '{snapshot_id}' AS snapshot_id, {seq_val} AS sequence_number, current_timestamp() AS updated_at) AS s
        ON t.source_table = s.source_table
        WHEN MATCHED THEN UPDATE SET t.snapshot_id = s.snapshot_id, t.sequence_number = s.sequence_number, t.updated_at = s.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)


results = []

for ns_name, table_name in all_tables:
    full_path = f"{ns_name}.{table_name}"
    target_table_name = full_path.replace(".", "_")

    print(f"\n{'='*60}")
    print(f"Processing: {full_path} -> {target_table_name}")
    print(f"{'='*60}")

    try:
        t0 = time.time()
        iceberg_table = catalog.load_table(full_path)

        # Find time column
        time_col_name, is_tz = None, False
        for field in iceberg_table.schema().fields:
            if field.name in ("time", "lastUpdated", "creationTime"):
                time_col_name, is_tz = field.name, isinstance(field.field_type, TimestamptzType)
                break

        watermark = get_watermark(target_table_name, time_col_name) if time_col_name else None

        # Snapshot-based short-circuit
        snap = iceberg_table.current_snapshot()
        snap_id = str(snap.snapshot_id) if snap else None
        snap_seq = snap.sequence_number if snap else None
        stored_id, stored_seq = get_stored_snapshot(target_table_name)

        if snap_id and snap_id == stored_id:
            print(f"  SKIPPED: Snapshot unchanged ({snap_id}). No S3 scan needed. ({time.time()-t0:.1f}s)")
            results.append({"table": full_path, "status": "skipped", "reason": "snapshot unchanged", "rows": 0})
            continue

        # Build PyIceberg scan with time filter (with fallback for equality deletes)
        tz_sfx = "+00:00" if is_tz else ""
        end_str = run_end_time.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] if watermark or time_col_name else None

        if watermark:
            print(f"  Watermark (last imported): {watermark}")
            print(f"  Fetching data from {watermark} to {run_end_time.isoformat()}")

        try:
            if watermark:
                row_filter = And(
                    GreaterThan(time_col_name, watermark + tz_sfx),
                    LessThanOrEqual(time_col_name, end_str + tz_sfx)
                )
                arrow_table = iceberg_table.scan(row_filter=row_filter).to_arrow()
            elif time_col_name:
                print(f"  No existing data \u2014 full initial load")
                arrow_table = iceberg_table.scan().to_arrow()
            else:
                print(f"  No time column \u2014 full load (table will be overwritten)")
                arrow_table = iceberg_table.scan().to_arrow()
            pdf = arrow_table.to_pandas()
        except Exception as scan_err:
            if "equality deletes" not in str(scan_err).lower():
                raise
            # Fallback: read new Parquet files directly (bypasses equality delete limitation)
            print(f"  Equality deletes detected \u2014 reading new files directly via PyArrow")
            pdf = _read_new_files(iceberg_table, stored_seq)
            # Apply time filter in pandas (since we bypassed PyIceberg's row_filter)
            if watermark and time_col_name and len(pdf) > 0:
                wm_ts = pd_lib.Timestamp(watermark + tz_sfx)
                end_ts = pd_lib.Timestamp(end_str + tz_sfx)
                pdf = pdf[(pdf[time_col_name] > wm_ts) & (pdf[time_col_name] <= end_ts)].reset_index(drop=True)
                print(f"  Filtered to {len(pdf)} rows within time range")

        row_count = len(pdf)

        if row_count == 0:
            if snap_id:
                store_snapshot(target_table_name, snap_id, snap_seq)
            print(f"  SKIPPED: No new data since last import. ({time.time()-t0:.1f}s)")
            results.append({"table": full_path, "status": "skipped", "reason": "no new data", "rows": 0})
            continue

        print(f"  Read {row_count} rows via PyIceberg")

        # Flatten and prepare
        pdf = flatten_pandas_df(pdf)
        for c in list(pdf.columns):
            if pdf[c].isna().all():
                pdf[c] = pdf[c].astype("object")

        df = spark.createDataFrame(pdf)

        # Build all column transforms at once (single withColumns call)
        schema_fields = df.schema.fields
        null_cols = {f.name for f in schema_fields if str(f.dataType) == "NullType()"}
        transforms = {}
        if null_cols:
            existing_schema = {}
            try:
                existing_schema = {f.name: f.dataType for f in spark.table(target_table_name).schema.fields}
            except Exception:
                pass
            transforms.update({name: lit(None).cast(existing_schema.get(name, StringType())) for name in null_cols})

        try:
            id_offset = spark.sql(f"SELECT COALESCE(MAX(row_id), -1) as max_id FROM {target_table_name}").collect()[0]["max_id"] + 1
        except Exception:
            id_offset = 0
        transforms["row_id"] = monotonically_increasing_id() + lit(id_offset).cast(LongType())
        df = df.withColumns(transforms)

        # Write
        write_mode = "append" if time_col_name else "overwrite"
        schema_opt = "mergeSchema" if time_col_name else "overwriteSchema"
        df.write.mode(write_mode).option(schema_opt, "true").saveAsTable(target_table_name)

        elapsed = time.time() - t0
        if snap_id:
            store_snapshot(target_table_name, snap_id, snap_seq)

        print(f"  SUCCESS: {row_count} rows {write_mode}ed to '{target_table_name}' ({elapsed:.1f}s, pyiceberg)")
        print(f"  Schema: {[f.name for f in schema_fields] + ['row_id']}")
        results.append({"table": full_path, "status": "success", "rows": row_count, "target": target_table_name, "reader": "pyiceberg", "mode": write_mode, "seconds": round(elapsed, 1)})

    except Exception as e:
        print(f"  ERROR: {str(e)[:300]}")
        results.append({"table": full_path, "status": "error", "reason": str(e)[:300], "rows": 0})

print(f"\nDone. Processed {len(results)} tables.")

Snapshot tracking table ready.
Run end time (upper bound): 2026-09-14T10:59:08.088410+00:00

Processing: cdc_alarm.alarm -> cdc_alarm_alarm
  SKIPPED: Snapshot unchanged (5250194044501598380). No S3 scan needed. (1.4s)

Processing: cdc_event.c8y_Position -> cdc_event_c8y_Position
  Watermark (last imported): 2026-09-14T10:52:23.641
  Fetching data from 2026-09-14T10:52:23.641 to 2026-09-14T10:59:08.088410+00:00
  Read 78 rows via PyIceberg
  SUCCESS: 78 rows appended to 'cdc_event_c8y_Position' (3.8s, pyiceberg)
  Schema: ['id', 'source', 'eventType', 'time', 'lng', 'alt', 'accuracy', 'lat', 'row_id']

Processing: cdc_event.event -> cdc_event_event
  SKIPPED: Snapshot unchanged (4743870334578038413). No S3 scan needed. (1.5s)

Processing: cdc_event.jobStatus -> cdc_event_jobStatus
  SKIPPED: Snapshot unchanged (4545562973280546472). No S3 scan needed. (1.3s)

Processing: cdc_inventory.c8y_ActiveAlarmsStatus -> cdc_inventory_c8y_ActiveAlarmsStatus
  SKIPPED: Snapshot unchanged (76141945

In [0]:
# 5. Print import summary
import pandas as pd

print(f"\n{'='*60}")
print(f"IMPORT SUMMARY")
print(f"{'='*60}")

success_count = sum(1 for r in results if r["status"] == "success")
skipped_count = sum(1 for r in results if r["status"] == "skipped")
error_count = sum(1 for r in results if r["status"] == "error")
total_rows = sum(r["rows"] for r in results)

print(f"Total tables processed: {len(results)}")
print(f"Successful: {success_count}")
print(f"Skipped (no new data): {skipped_count}")
print(f"Errors: {error_count}")
print(f"\nTotal new rows imported: {total_rows}")
print()

df_summary = pd.DataFrame(results)
df_summary


IMPORT SUMMARY
Total tables processed: 40
Successful: 3
Skipped (no new data): 37
Errors: 0

Total new rows imported: 706



,table,status,reason,rows,target,reader,mode,seconds
0,cdc_alarm.alarm,skipped,snapshot unchanged,0,NaN,NaN,NaN,NaN
1,cdc_event.c8y_Position,success,NaN,78,cdc_event_c8y_Position,pyiceberg,append,3.8
2,cdc_event.event,skipped,snapshot unchanged,0,NaN,NaN,NaN,NaN
3,cdc_event.jobStatus,skipped,snapshot unchanged,0,NaN,NaN,NaN,NaN
4,cdc_inventory.c8y_ActiveAlarmsStatus,skipped,snapshot unchanged,0,NaN,NaN,NaN,NaN
5,cdc_inventory.c8y_DeviceSimulator,skipped,snapshot unchanged,0,NaN,NaN,NaN,NaN
6,cdc_inventory.c8y_Position,skipped,snapshot unchanged,0,NaN,NaN,NaN,NaN
7,cdc_inventory.c8y_ua_Node,skipped,no new data,0,NaN,NaN,NaN,NaN
8,cdc_inventory.com_cumulocity_opcua_common_mode...,skipped,no new data,0,NaN,NaN,NaN,NaN
9,cdc_inventory.inventory,skipped,snapshot unchanged,0,NaN,NaN,NaN,NaN


In [0]:
# 6. Ensure primary key constraints on all target tables
success_tables = [r for r in results if r["status"] == "success"]
target_names = list({r["target"] for r in success_tables})

pk_count = 0
for tname in sorted(target_names):
    constraint_name = f"pk_{tname}"
    try:
        spark.sql(f"ALTER TABLE {tname} ADD CONSTRAINT {constraint_name} PRIMARY KEY (row_id)")
        print(f"  \u2713 {tname}: PRIMARY KEY added")
        pk_count += 1
    except Exception as e:
        if "already exists" in str(e).lower() or "CONSTRAINT_ALREADY_EXISTS" in str(e):
            print(f"  \u2713 {tname}: PRIMARY KEY already exists")
            pk_count += 1
        else:
            print(f"  \u2717 {tname}: {str(e)[:120]}")

print(f"\nPrimary keys ensured: {pk_count} / {len(target_names)}")


Primary keys ensured: 0 / 0
